# SecureFinger — reviewer experiments on Colab (~45 min)
Runtime → **T4 GPU**. On the Mac keep `caffeinate -d` running.

* **(e)** SourceAFIS minutiae baseline on the same FVC pairs (~10 min, CPU)
* **(f)** fine-tuning recipe chosen on held-out data (nested selection, ~30 min, GPU)

## 1. Upload the NEW `securefinger.zip` and install

In [ ]:
from google.colab import files
files.upload()  # choose securefinger.zip
!unzip -q -o securefinger.zip
%cd /content/securefinger
!pip -q install tenseal fastapi httpx cryptography opencv-python-headless pyjwt scikit-learn

## 2. Drive (keeps results if Colab disconnects) + encoder weights

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/securefinger_results_review'
!mkdir -p "$OUT"
up = files.upload()   # choose encoder_seed0.pt
WEIGHTS = '/content/securefinger/' + list(up)[0]
!ls -la "$WEIGHTS"
!nvidia-smi --query-gpu=name --format=csv

## 3. Java + SourceAFIS (~2 min)

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless maven > /dev/null
!java -version
%cd /content/securefinger/tools/sourceafis
!mvn -q dependency:copy-dependencies -DoutputDirectory=lib && javac -cp "lib/*" Match.java && ls lib | head && echo COMPILED_OK
%cd /content/securefinger

## 4. (e) Minutiae baseline (~10 min)
Downloads the FVC + Neurotechnology data again, then scores every pair with SourceAFIS.

In [ ]:
!python experiments/minutiae_baseline.py --root /content/fvc --out "$OUT/minutiae" 2>&1 | tee "$OUT/minutiae_log.txt"

## 5. (f) Nested recipe selection (~30 min)

In [ ]:
!python experiments/fvc_nested.py --root /content/fvc --no-download --weights "$WEIGHTS" --out "$OUT/nested" 2>&1 | tee "$OUT/nested_log.txt"

## 6. Summary to paste back

In [ ]:
import json, os
p = f'{OUT}/minutiae/minutiae.json'
if os.path.exists(p):
    r = json.load(open(p)); a = r['all_fingers']['pooled_real']
    print('== SourceAFIS all fingers pooled: EER %.2f%%  FMR100 %.2f%%' % (a['eer']*100, a['fnmr_at_fmr_1e-2']*100))
    for k, v in r['cv_test_folds']['summary'].items(): print(f'   CV test folds {k:18s} {v["mean"]*100:6.2f} ± {v["std"]*100:.2f} %')
    for d, m in r['all_fingers']['per_db'].items(): print(f'   {d:16s} EER {m["eer"]*100:6.2f}%')
p = f'{OUT}/nested/nested.json'
if os.path.exists(p):
    r = json.load(open(p)); print('== nested selection', f"{r.get('minutes', 0):.1f} min", r.get('chosen_counts'))
    for k, v in r.get('summary', {}).items(): print(f'   {k:18s} {v["mean"]*100:6.2f} ± {v["std"]*100:.2f} %')
    for run in r['runs']:
        for f in run['folds']: print('   seed', run['seed'], 'fold', f['fold'], 'chose', f['chosen'], {k: round(v*100, 2) for k, v in f['inner_val_eer'].items()}, 'test %.2f%%' % (f['test_fold_eer']*100))

## 7. Download

In [ ]:
!cd "$OUT" && zip -q -r /content/results_review.zip . -x 'minutiae/png/*'
from google.colab import files
files.download('/content/results_review.zip')